In [7]:
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
from torch.nn import functional as F
import torch
import pandas as pd
#import wfdb
from scipy.stats import pearsonr
#from SoftDTW_functions import SoftDTW
#from tqdm import tqdm
import multiprocessing as mp
from matplotlib.cm import get_cmap
import neurokit2 as nk
import ast


from multiprocessing import Pool, cpu_count
from functools import partial

#from Load_Data import load_ptb_xl_data
from VQ_VAE import Encoder_net, Quantize_net, Decoder_net
from NanoGPT import GPTLanguageModel

In [67]:
def decode_embed(emb_ind, codebook):
    """
    Decodes embedding indices into their corresponding codebook vectors.
    Avoids .numpy() calls for compatibility with MPS (Apple Silicon) backend.

    Args:
        emb_ind (torch.Tensor): Embedding indices of shape (batch, seq_len).
        codebook (torch.Tensor): Codebook vectors of shape (encoding_size, vocab_size).

    Returns:
        torch.Tensor: Decoded embeddings of shape (batch, encoding_size, seq_len).
    """
    # Move indices to CPU if they are a PyTorch tensor
    if isinstance(emb_ind, torch.Tensor):
        emb_ind = emb_ind.detach().cpu()
    codebook = codebook.detach().cpu()
    
    new_out = []
    for i in range(len(emb_ind)):
        indices = emb_ind[i].long()
        # Vectorized lookup: select one codebook vector per index
        vectors = codebook[:, indices]  # shape: (encoding_size, seq_len)
        new_out.append(vectors)
    
    # Stack all samples into a single batch tensor
    return torch.stack(new_out)  # shape: (batch, encoding_size, seq_len)



def generate_ecg(encoder, decoder, quantizer, NanoGPT, nb_ecg, device, ecg, block_size=77):
    """
    Generates synthetic ECG signals using a VQ-VAE + GPT pipeline.

    The generation process:
        1. Encode the input ECG with the encoder.
        2. Quantize the encoded representation to get discrete token indices.
        3. Use NanoGPT to autoregressively generate new token sequences.
        4. Decode the generated tokens back into continuous embeddings.
        5. Reconstruct the ECG signal using the decoder.

    Args:
        encoder: VQ-VAE encoder model.
        decoder: VQ-VAE decoder model.
        quantizer: Vector quantization module; returns (quantized, loss, indices, codebook).
        NanoGPT: Autoregressive GPT model for token sequence generation.
        nb_ecg (int): Number of ECG generation iterations to run.
        device (torch.device): Target device (CPU, CUDA, or MPS).
        ecg (torch.Tensor): Seed ECG tensor used as generation context.
        block_size (int): Number of initial tokens fed to NanoGPT as context. Default: 77.

    Returns:
        list[torch.Tensor]: List of generated ECG tensors, each on CPU.
    """
    print("Generating ECGs...", str(block_size))
    batch_size = 256
    
    generate_ecgs = []
    for i in range(nb_ecg):
        # --- Case 1: input fits in a single batch ---
        if len(ecg) < batch_size:
            inp = ecg
            with torch.no_grad():
                block_size = 77
                inp = torch.tensor(inp, dtype=torch.float32).to(device)

                # Step 1: Encode the input ECG
                encoded = encoder(inp)

                # Step 2: Quantize → get discrete token indices and codebook
                quantized, _, embed_ind, codebook = quantizer(encoded.transpose(1, 2))

                # Step 3: Generate new token sequence with NanoGPT
                # Only the first block_size tokens are used as context
                generated = NanoGPT.generate(embed_ind[:, :block_size], max_new_tokens=306)

                generated = generated.detach().cpu()
                ind_generated = generated  # Save raw indices for inspection if needed

                # Step 4: Decode token indices back into continuous embeddings
                generated = decode_embed(generated, codebook.detach().cpu())

                # Step 5: Reconstruct the ECG signal from embeddings
                generated = decoder(generated)
                generated = generated.detach().cpu()

            generate_ecgs.append(generated)

        # --- Case 2: input exceeds batch size → process in mini-batches ---
        else:
            for b in range(0, len(ecg), batch_size):
                inp = ecg[b:b + batch_size]
                with torch.no_grad():
                    inp = torch.tensor(inp, dtype=torch.float32).to(device)

                    # Step 1: Encode the mini-batch
                    encoded = encoder(inp)

                    # Step 2: Quantize → get discrete token indices and codebook
                    quantized, _, embed_ind, codebook = quantizer(encoded.transpose(1, 2))

                    # Step 3: Generate new token sequence with NanoGPT
                    generated = NanoGPT.generate(embed_ind[:, :block_size], max_new_tokens=306)

                    generated = generated.detach().cpu()
                    ind_generated = generated  # Save raw indices for inspection if needed

                    # Step 4: Decode token indices back into continuous embeddings
                    generated = decode_embed(generated, codebook.detach().cpu())

                    # Step 5: Reconstruct the ECG signal from embeddings
                    generated = decoder(generated)
                    generated = generated.detach().cpu()

                generate_ecgs.append(generated)

    return generate_ecgs

In [68]:
# ── Configuration ─────────────────────────────────────────────────────────────

vocab_size = 64

# Dynamically select the best available device (GPU > CPU, MPS not used due to numpy unavailability)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ── Model Initialization ───────────────────────────────────────────────────────

# Initialize the encoder with 4 downsampling levels
encoder = Encoder_net(4)

# Perform a dry run to infer the encoder's output size dynamically
# This avoids hardcoding the encoding dimension
with torch.no_grad():
    temp = torch.randn(1, 1, 5000)  # Dummy input: (batch=1, channels=1, length=5000)
    temp = encoder(temp)
    encoding_size = temp.shape[1]   # Extract the number of encoded channels

# Initialize decoder with one fewer level than encoder, using the inferred encoding size
decoder = Decoder_net(4 - 1, chanel_in=encoding_size)

# Initialize the vector quantizer with the encoding size and the codebook vocabulary size
quantizer = Quantize_net(encoding_size, vocab_size)

# Initialize the GPT language model for autoregressive token generation
NanoGPT = GPTLanguageModel(
    block_size=306,      # Maximum sequence length the model can handle
    vocab_size=vocab_size,
    n_embd=384,          # Embedding dimension
    n_layer=10,          # Number of transformer layers
    n_head=10,           # Number of attention heads
    dropout=0.2,
    num_classes=12,      # Number of ECG lead classes
    device=device
)

# ── Utility: Model Loader ──────────────────────────────────────────────────────

def load_model(model, path, device):
    """
    Loads a model's state dict from disk and prepares it for inference.

    Args:
        model (nn.Module): The model instance to load weights into.
        path (str): Path to the .pth file containing the state dict.
        device (torch.device): Device to map the weights onto.

    Returns:
        nn.Module: The model loaded with pretrained weights, in eval mode.
    """
    state_dict = torch.load(path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()  # Disable dropout and batch norm tracking for inference
    return model

# ── Model Paths ────────────────────────────────────────────────────────────────

encoder_path   = f"../Models/VQ-VAE/Encoder_vocab_size_{vocab_size}_4_epoch0.pth"
decoder_path   = f"../Models/VQ-VAE/Decoder_vocab_size_{vocab_size}_4_epoch0.pth"
quantizer_path = f"../Models/VQ-VAE/Quantizer_vocab_size_{vocab_size}_4_epoch0.pth"
gpt_path       = f"../Models/NanoGPT/Transformer_{vocab_size}_10.pth"

# ── Load Pretrained Weights ────────────────────────────────────────────────────

encoder   = load_model(encoder,   encoder_path,   device)
decoder   = load_model(decoder,   decoder_path,   device)
quantizer = load_model(quantizer, quantizer_path, device)
NanoGPT   = load_model(NanoGPT,   gpt_path,       device)

# ── ECG Generation ────────────────────────────────────────────────────────────

# Generate 10 synthetic ECGs using the first 2 samples of the subset as seed context
# block_size=77 means only the first 77 tokens are used as GPT context
generated_ecgs = generate_ecg(
    encoder, decoder, quantizer, NanoGPT,
    nb_ecg=10,
    device=device,
    ecg=torch.tensor(subset[:2]).to(device),
    block_size=77
)

Using device: cpu
